# Поиск похожих пользователей через FAISS

В этом ноутбуке мы последовательно:

1. Загружаем датасет `data/user_features_data1.csv`.
2. Готовим признаки для поиска похожих пользователей.
3. Нормализуем признаки с помощью `RobustScaler`.
4. Строим ANN-индекс на базе **FAISS** с метрикой **Euclidean distance**.
5. Реализуем функцию `get_n_similarest`.
6. Возвращаем ближайших пользователей, их расстояния, ранг, `ID` и фичи.
7. Для блоков `cat_share_*` и `mcc_share_*` добавляем удобное обратное представление.

Пояснения оставлены подробными, но сам код разбит на отдельные функции и не перегружен лишними деталями.

## 1. Установка зависимостей

Если библиотеки уже установлены в текущем Jupyter-ядре, эта ячейка просто сообщит об этом.

In [1]:
import importlib
import subprocess
import sys


def ensure_package(package_name, module_name=None):
    module_name = module_name or package_name
    try:
        importlib.import_module(module_name)
        print(f"{package_name}: уже установлен")
    except ModuleNotFoundError:
        print(f"{package_name}: начинаю установку")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"{package_name}: установка завершена")


for package_name, module_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scikit-learn", "sklearn"),
    ("faiss-cpu", "faiss"),
]:
    ensure_package(package_name, module_name)

numpy: уже установлен
pandas: уже установлен
scikit-learn: уже установлен
faiss-cpu: уже установлен


## 2. Импорты и основные константы

Имена `CandidateConfig` и `get_n_similarest` сохранены по постановке задачи.
Код написан в привычном Python-стиле, а пояснения и пользовательские сообщения оставлены на русском языке.

In [2]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

try:
    import faiss
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Модуль faiss не найден. Сначала выполните ячейку с установкой зависимостей."
    ) from exc


pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)
np.random.seed(42)

DATA_PATH = Path("data/user_features_data1.csv")
ID_COLUMN = "party_rk"
DATE_COLUMNS = ["first_txn_dttm", "last_txn_dttm"]
TOP_CATEGORY_COLUMN = "top_cat_from_share"
TOP_CATEGORY_SHARE_COLUMN = "top_cat_share"
TOP_MCC_COLUMN = "top_mcc_from_share"
TOP_MCC_SHARE_COLUMN = "top_mcc_share"


@dataclass(frozen=True)
class CandidateConfig:
    n_rec_users: int = 5
    n_min_candidates: int = 100
    topk_trans_sim: int = 150
    topk_inter_sim: int = 150
    trans_inter_rate: float = 0.5

## 3. Загрузка данных

Датасет довольно большой, поэтому сначала читаем небольшой сэмпл и определяем:

- какие колонки являются датами;
- какие колонки строковые;
- какие числовые типы разумно использовать при полной загрузке.

Такой шаг помогает аккуратнее контролировать память.

In [3]:
def infer_load_config(file_path, sample_rows=2000):
    sample_df = pd.read_csv(file_path, nrows=sample_rows)

    date_columns = [column for column in DATE_COLUMNS if column in sample_df.columns]
    categorical_columns = [
        column
        for column in sample_df.columns
        if column not in [ID_COLUMN, *date_columns]
        and not pd.api.types.is_numeric_dtype(sample_df[column])
    ]

    dtype_map = {ID_COLUMN: "string"}
    for column in sample_df.columns:
        if column in [ID_COLUMN, *date_columns, *categorical_columns]:
            continue

        if pd.api.types.is_integer_dtype(sample_df[column]):
            dtype_map[column] = "Int32"
        elif pd.api.types.is_float_dtype(sample_df[column]):
            dtype_map[column] = "float32"

    for column in categorical_columns:
        dtype_map[column] = "string"

    return dtype_map, date_columns, categorical_columns


def load_dataset(file_path):
    dtype_map, date_columns, categorical_columns = infer_load_config(file_path)

    df = pd.read_csv(
        file_path,
        dtype=dtype_map,
        parse_dates=date_columns,
        low_memory=False,
    )

    for column in categorical_columns:
        df[column] = df[column].astype("string")

    return df


users_df = load_dataset(DATA_PATH)

print(f"Файл найден: {DATA_PATH.exists()}")
print(f"Размер таблицы: {users_df.shape}")
print(f"Уникальных пользователей: {users_df[ID_COLUMN].nunique():,}")

display(users_df.head(3))

Файл найден: True
Размер таблицы: (1217027, 98)
Уникальных пользователей: 1,217,027


,party_rk,txn_cnt,unique_transaction_cnt,active_days_cnt,active_months_cnt,active_weeks_cnt,first_txn_dttm,last_txn_dttm,activity_span_days,avg_txn_per_active_day,avg_txn_per_month_active,avg_txn_per_week_active,dominant_hour,dominant_weekday_num,share_weekend_txn,share_weekday_txn,share_night_txn,share_morning_txn,share_afternoon_txn,share_evening_txn,dominant_daypart,max_txn_in_one_day,median_txn_per_active_day,unique_category_cnt,top_category_txn_share,category_entropy,cat_share_супермаркеты,cat_share_фаст_фуд,cat_share_транспорт,cat_share_разные_товары,cat_share_топливо,cat_share_рестораны,cat_share_аптеки,cat_share_дом_ремонт,cat_share_развлечения,cat_share_сервисные_услуги,cat_share_автоуслуги,cat_share_связь_телеком,cat_share_одежда_обувь,cat_share_красота,cat_share_медицинские_услуги,cat_share_госсборы,cat_share_аренда_авто,cat_share_частные_услуги,cat_share_нко,cat_share_финансовые_услуги,cat_share_цветы,cat_share_книги,cat_share_животные,cat_share_образование,cat_share_кино,cat_share_турагентства,cat_share_ж_д_билеты,cat_share_авиабилеты,cat_share_спорттовары,cat_share_unknown,cat_share_other,unique_mcc_cnt,top_mcc_txn_share,mcc_entropy,mcc_share_5411,mcc_share_5814,mcc_share_5499,mcc_share_3990,mcc_share_4131,mcc_share_5912,mcc_share_5541,mcc_share_5921,mcc_share_5812,mcc_share_4111,mcc_share_5300,mcc_share_4121,mcc_share_5200,mcc_share_7995,mcc_share_5331,mcc_share_5993,mcc_share_7542,mcc_share_5964,mcc_share_4900,mcc_share_5816,mcc_share_3991,mcc_share_5815,mcc_share_5311,mcc_share_5462,mcc_share_7512,mcc_share_4814,mcc_share_4816,mcc_share_5399,mcc_share_5977,mcc_share_5533,mcc_share_unknown,mcc_share_other,unique_merchant_cnt,top_merchant_txn_share,dominant_weekday,merchant_diversity_ratio,category_diversity_ratio,mcc_diversity_ratio
0,07e95d24bb488b49697420c27d035110259f37ccb1c3fddb01c8b6cd8ae4ee13,2,2,1,1,1,2025-02-03 14:17:04,2025-02-03 14:28:39,0,2.000000,2.0,2.0,14,1,0.000000,1.000000,0.0,0.00000,1.000000,0.000000,afternoon,2,2.0,1,1.00000,-0.000000,1.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1,1.00000,-0.000000,1.000000,0.00000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,2,0.500000,Monday,1.000000,0.500000,0.50000
1,cb450aa7466b337776f0f76debb5a485da9afeb7f49ec5d35481d65b8b822e31,62,62,21,1,5,2025-02-01 16:32:20,2025-02-28 22:11:01,27,2.952381,62.0,12.4,19,4,0.274194,0.725806,0.0,0.16129,0.435484,0.403226,afternoon,9,2.0,14,0.16129,2.386007,0.16129,0.16129,0.145161,0.048387,0.080645,0.112903,0.032258,0.0,0.0,0.0,0.032258,0.016129,0.064516,0.0,0.016129,0.064516,0.0,0.016129,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.048387,0.0,21,0.16129,2.649669,0.145161,0.16129,0.0,0.016129,0.0,0.032258,0.064516,0.0,0.112903,0.129032,0.016129,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016129,0.0,0.0,0.0,0.0,0.016129,0.0,0.032258,0.0,0.032258,0.0,0.225806,36,0.129032,Thursday,0.580645,0.225806,0.33871
2,826ea969a9b6ddc9de5c48bc0cea7e3fae27a340c95dcc329c079caf987c593e,4,4,3,1,2,2025-02-11 06:53:44,2025-02-27 07:11:47,16,1.333333,4.0,2.0,6,2,0.000000,1.000000,0.0,0.75000,0.250000,0.000000,morning,2,1.0,1,1.00000,-0.000000,1.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1,1.00000,-0.000000,0.000000,0.00000,1.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,2,0.750000,Tuesday,0.500000,0.250000,0.25000


## 4. Подготовка признаков

На этом шаге мы:

- переводим даты в числовые признаки;
- добавляем интерпретируемые колонки для `cat_share_*` и `mcc_share_*`;
- кодируем строковые признаки;
- заполняем пропуски;
- нормализуем матрицу признаков через `RobustScaler`.

Важно: сами исходные долевые признаки мы не теряем. Мы только добавляем поверх них удобные поля `top_cat_from_share` и `top_mcc_from_share`.

In [4]:
def add_date_features(df):
    result = df.copy()
    available_date_columns = [column for column in DATE_COLUMNS if column in result.columns]

    if not available_date_columns:
        return result

    reference_date = max(result[column].max() for column in available_date_columns)

    if "first_txn_dttm" in result.columns:
        result["days_since_first_txn"] = (
            reference_date - result["first_txn_dttm"]
        ).dt.days.astype("float32")

    if "last_txn_dttm" in result.columns:
        result["days_since_last_txn"] = (
            reference_date - result["last_txn_dttm"]
        ).dt.days.astype("float32")

    if {"first_txn_dttm", "last_txn_dttm"}.issubset(result.columns):
        result["activity_length_days"] = (
            result["last_txn_dttm"] - result["first_txn_dttm"]
        ).dt.days.astype("float32")

    return result


def decode_top_share_feature(df, prefix, label_column, share_column):
    result = df.copy()
    share_columns = [column for column in result.columns if column.startswith(prefix)]

    if not share_columns:
        return result

    share_frame = result[share_columns].fillna(0.0).astype("float32")
    share_matrix = share_frame.to_numpy()
    max_positions = share_matrix.argmax(axis=1)
    max_values = share_matrix[np.arange(len(result)), max_positions]
    share_sums = share_matrix.sum(axis=1)

    labels = [share_columns[position].replace(prefix, "") for position in max_positions]
    labels = np.where(share_sums > 0, labels, "не_определено")
    max_values = np.where(share_sums > 0, max_values, np.nan)

    result[label_column] = labels
    result[share_column] = max_values
    return result


def add_interpretable_share_columns(df):
    result = df.copy()
    result = decode_top_share_feature(
        result,
        prefix="cat_share_",
        label_column=TOP_CATEGORY_COLUMN,
        share_column=TOP_CATEGORY_SHARE_COLUMN,
    )
    result = decode_top_share_feature(
        result,
        prefix="mcc_share_",
        label_column=TOP_MCC_COLUMN,
        share_column=TOP_MCC_SHARE_COLUMN,
    )
    return result


def prepare_search_artifacts(df):
    enriched_df = add_date_features(df)
    output_df = add_interpretable_share_columns(enriched_df).reset_index(drop=True)

    available_date_columns = [column for column in DATE_COLUMNS if column in enriched_df.columns]
    model_df = enriched_df.drop(columns=available_date_columns).copy()

    categorical_columns = []
    for column in model_df.columns:
        if column == ID_COLUMN:
            continue

        is_object = pd.api.types.is_object_dtype(model_df[column])
        is_string = pd.api.types.is_string_dtype(model_df[column])
        is_category = isinstance(model_df[column].dtype, pd.CategoricalDtype)

        if is_object or is_string or is_category:
            categorical_columns.append(column)

    encoded_df = pd.get_dummies(
        model_df,
        columns=categorical_columns,
        dummy_na=False,
        dtype=np.float32,
    )

    feature_df = encoded_df.drop(columns=[ID_COLUMN]).apply(pd.to_numeric, errors="coerce")

    imputer = SimpleImputer(strategy="median")
    imputed_matrix = imputer.fit_transform(feature_df).astype("float32")

    scaler = RobustScaler()
    scaled_matrix = scaler.fit_transform(imputed_matrix).astype("float32")

    return {
        "output_df": output_df,
        "feature_df": feature_df,
        "imputer": imputer,
        "scaler": scaler,
        "scaled_matrix": np.ascontiguousarray(scaled_matrix),
        "user_ids": output_df[ID_COLUMN].astype("string").to_numpy(),
        "id_index": pd.Index(output_df[ID_COLUMN].astype("string"), name=ID_COLUMN),
    }


matrix_artifacts = prepare_search_artifacts(users_df)

print("Матрица признаков подготовлена.")
print(f"Исходных колонок: {users_df.shape[1]}")
print(f"Колонок после кодирования: {matrix_artifacts['feature_df'].shape[1]}")
print(f"Форма итоговой матрицы: {matrix_artifacts['scaled_matrix'].shape}")

preview_columns = [
    ID_COLUMN,
    TOP_CATEGORY_COLUMN,
    TOP_CATEGORY_SHARE_COLUMN,
    TOP_MCC_COLUMN,
    TOP_MCC_SHARE_COLUMN,
]
display(matrix_artifacts["output_df"][preview_columns].head(3))

Матрица признаков подготовлена.
Исходных колонок: 98
Колонок после кодирования: 107
Форма итоговой матрицы: (1217027, 107)


,party_rk,top_cat_from_share,top_cat_share,top_mcc_from_share,top_mcc_share
0,07e95d24bb488b49697420c27d035110259f37ccb1c3fddb01c8b6cd8ae4ee13,супермаркеты,1.00000,5411,1.000000
1,cb450aa7466b337776f0f76debb5a485da9afeb7f49ec5d35481d65b8b822e31,супермаркеты,0.16129,other,0.225806
2,826ea969a9b6ddc9de5c48bc0cea7e3fae27a340c95dcc329c079caf987c593e,супермаркеты,1.00000,5499,1.000000


## 5. Построение FAISS-индекса

Используем `faiss.IndexHNSWFlat`:

- это ANN-индекс, то есть он ускоряет поиск по большому числу пользователей;
- внутри используется L2-метрика;
- после получения кандидатов мы дополнительно пересчитываем **точное евклидово расстояние** и заново сортируем результат.

Так мы совмещаем скорость ANN и прозрачность итогового ранжирования.

In [5]:
def build_faiss_index(scaled_matrix, config):
    feature_count = scaled_matrix.shape[1]
    index = faiss.IndexHNSWFlat(feature_count, 32, faiss.METRIC_L2)
    index.hnsw.efConstruction = 200
    index.hnsw.efSearch = max(config.topk_trans_sim * 2, config.n_min_candidates, 200)
    index.add(np.ascontiguousarray(scaled_matrix))
    return index


base_config = CandidateConfig()
faiss_index = build_faiss_index(matrix_artifacts["scaled_matrix"], base_config)

search_artifacts = {
    **matrix_artifacts,
    "faiss_index": faiss_index,
}

print("FAISS-индекс построен.")
print(f"Количество векторов в индексе: {faiss_index.ntotal}")

FAISS-индекс построен.
Количество векторов в индексе: 1217027


## 6. Функция `get_n_similarest`

Логика функции такая:

1. Берём `user_id`.
2. Ищем расширенный набор кандидатов через FAISS.
3. Убираем самого пользователя из выдачи.
4. Пересчитываем точные Euclidean distance по нормализованной матрице.
5. Возвращаем ближайших пользователей с рангом, расстоянием, `ID` и фичами.

Параметр `topk_trans_sim` используется как глубина поиска кандидатов в транзакционном пространстве. Остальные поля конфига сохранены для совместимости с вашей исходной структурой.

In [6]:
def get_n_similarest(user_id, config=CandidateConfig(), search_bundle=None):
    if search_bundle is None:
        if "search_artifacts" not in globals():
            raise ValueError("Сначала подготовьте search_artifacts и FAISS-индекс.")
        search_bundle = search_artifacts

    user_id = str(user_id)
    id_index = search_bundle["id_index"]

    if user_id not in id_index:
        raise KeyError(f"Пользователь {user_id} не найден в датасете.")

    query_position = int(id_index.get_loc(user_id))
    scaled_matrix = search_bundle["scaled_matrix"]
    index = search_bundle["faiss_index"]
    query_vector = scaled_matrix[query_position : query_position + 1]

    candidate_count = min(
        len(scaled_matrix),
        max(
            config.topk_trans_sim,
            config.n_min_candidates,
            config.n_rec_users + 1,
        ),
    )

    _, raw_candidate_indices = index.search(query_vector, candidate_count + 1)
    raw_candidate_indices = raw_candidate_indices[0]

    unique_candidate_indices = []
    seen_indices = set()
    for candidate_index in raw_candidate_indices:
        candidate_index = int(candidate_index)
        if candidate_index < 0 or candidate_index == query_position:
            continue
        if candidate_index in seen_indices:
            continue
        seen_indices.add(candidate_index)
        unique_candidate_indices.append(candidate_index)

    if not unique_candidate_indices:
        raise ValueError("FAISS не вернул подходящих кандидатов для этого пользователя.")

    candidate_indices = np.asarray(unique_candidate_indices, dtype=np.int32)
    exact_distances = np.linalg.norm(
        scaled_matrix[candidate_indices] - query_vector[0],
        axis=1,
    )

    top_order = np.argsort(exact_distances)[: config.n_rec_users]
    top_indices = candidate_indices[top_order]
    top_distances = exact_distances[top_order]

    result_df = search_bundle["output_df"].iloc[top_indices].copy().reset_index(drop=True)
    result_df.insert(0, "rank", np.arange(1, len(result_df) + 1))
    result_df.insert(1, "distance_euclidean", top_distances.astype(float))
    result_df.insert(2, "query_party_rk", user_id)

    return result_df

## 7. Пример использования

Возьмём первого пользователя из датасета, найдём для него ближайших соседей и покажем компактную витрину результата.

In [7]:
example_user_id = users_df[ID_COLUMN].iloc[0]
result_df = get_n_similarest(
    example_user_id,
    config=CandidateConfig(n_rec_users=5, topk_trans_sim=150),
)

view_columns = [
    "rank",
    "distance_euclidean",
    "query_party_rk",
    "party_rk",
    "txn_cnt",
    "active_days_cnt",
    "unique_category_cnt",
    "unique_mcc_cnt",
    "dominant_daypart",
    "dominant_weekday",
    TOP_CATEGORY_COLUMN,
    TOP_CATEGORY_SHARE_COLUMN,
    TOP_MCC_COLUMN,
    TOP_MCC_SHARE_COLUMN,
]

print(f"Пример пользователя: {example_user_id}")
display(result_df[view_columns])

Пример пользователя: 07e95d24bb488b49697420c27d035110259f37ccb1c3fddb01c8b6cd8ae4ee13


,rank,distance_euclidean,query_party_rk,party_rk,txn_cnt,active_days_cnt,unique_category_cnt,unique_mcc_cnt,dominant_daypart,dominant_weekday,top_cat_from_share,top_cat_share,top_mcc_from_share,top_mcc_share
0,1,0.000000,07e95d24bb488b49697420c27d035110259f37ccb1c3fddb01c8b6cd8ae4ee13,7e0aaba1747809b640ad63c498347a95d9e4edf4d2b4e79412613b6aa34d8884,2,1,1,1,afternoon,Monday,супермаркеты,1.0,5411,1.0
1,2,0.000000,07e95d24bb488b49697420c27d035110259f37ccb1c3fddb01c8b6cd8ae4ee13,ca300d8ea69a10555211ece525c789d24ecb3223c51d61d39e2d44fefd10e5e1,2,1,1,1,afternoon,Monday,супермаркеты,1.0,5411,1.0
2,3,0.000000,07e95d24bb488b49697420c27d035110259f37ccb1c3fddb01c8b6cd8ae4ee13,b5fe4f725b717850b78bc9a8284d6f12cac1ca66f3844b39a0cbe4453c70ed5a,2,1,1,1,afternoon,Monday,супермаркеты,1.0,5411,1.0
3,4,0.000000,07e95d24bb488b49697420c27d035110259f37ccb1c3fddb01c8b6cd8ae4ee13,67ec6dd76b608b02b71c3c7c2cf045db584ea7079de8550af283e851b2ffc147,2,1,1,1,afternoon,Monday,супермаркеты,1.0,5411,1.0
4,5,0.166667,07e95d24bb488b49697420c27d035110259f37ccb1c3fddb01c8b6cd8ae4ee13,d7403a55e3d5a5750c6c2504266370d8514d884755ba4b2cb3bf56254c3a6ce5,2,1,1,1,afternoon,Monday,супермаркеты,1.0,5411,1.0


## 8. Итог

Функция `get_n_similarest` возвращает `pandas.DataFrame`, в котором уже есть:

- `rank` — позиция соседа в выдаче;
- `distance_euclidean` — итоговое евклидово расстояние;
- `query_party_rk` — пользователь, для которого выполнялся поиск;
- `party_rk` и остальные колонки датасета — фичи найденного соседа;
- `top_cat_from_share` и `top_mcc_from_share` — удобное обратное представление долевых признаков.

При желании вы можете оставить полный набор колонок или выбрать только нужные поля для витрины, отчёта или следующего этапа рекомендательной логики.